# Phase A Visual Boundary Model (B0.5)

**An interactive visualization of the *existing* reduced-order radiator-boundary
model** in the `orbital_thermal` package. Every number and curve below is produced
by an `orbital_thermal` engine function through the thin
`orbital_thermal.visual_api` orchestration layer &mdash; **no physics is
reimplemented in this notebook.**

> ### Scope and warning
> * This is a **verification / explanation** artifact for the Phase A radiator
>   *boundary* model (radiative equilibrium, required area, net rejection, exact
>   Earth view factor, effective sink, orbital transient).
> * It is **NOT flight validation** and **NOT** a design/certification tool. The
>   package provides mathematical, computational, software, and cross-model
>   verification of a one-node reduced-order model &mdash; not validation against
>   flown hardware.
> * It is **NOT a Phase B transport model**. Chip-to-radiator pumped-loop
>   transport is out of scope here and waits for Phase B (B4/B5) after the coupled
>   solver exists.
> * Cross-model comparison (e.g. McCalip) is a **replication + correction**
>   exercise, not qualified external human validation.

Each input shown is labelled by provenance
(`published` / `derived` / `assumed` / `corrected` / `design-variable` /
`sensitivity` / `unsupported/future`), and every plot carries units and the model
conventions / limitations it rests on.

## 2. Imports and environment check

Import the local package and confirm the engine entry points the notebook depends
on are all importable. Prints the package version and (if available) the git
commit, so a reviewer can pin exactly what produced these figures.

In [ ]:
import subprocess
import warnings

import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact
from IPython.display import HTML, display

import orbital_thermal as ot
from orbital_thermal import transient as _transient
from orbital_thermal import visual_api as vapi

info = vapi.engine_info()
try:
    _commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], text=True, stderr=subprocess.DEVNULL
    ).strip()
except Exception:
    _commit = "unknown (not a git checkout)"

print(f"orbital_thermal version : {info['version']}")
print(f"git commit              : {_commit}")
print(f"all engine functions ok : {info['all_functions_available']}")
for _name, _ok in info["functions"].items():
    print(f"   [{'ok ' if _ok else 'MISS'}] {_name}")

assert info["all_functions_available"], "required engine functions are missing"

## 3. Baseline reproduction

Reproduce selected **existing** Phase A outputs (the same anchors the package test
suite asserts) and check each within tolerance. These are engine values surfaced
through `visual_api`; if any line fails, the environment does not match the
verified package and the rest of the notebook should not be trusted.

| anchor | source |
| --- | --- |
| AI1 sustained 337.10 K, peak 353.16 K | `equilibrium.equilibrium_temperature` (companion paper, doi:10.5281/zenodo.20670771) |
| 1 MW @ 293 K, zero sink &rarr; 2630 m&sup2; emitting | `radiation.required_area` (Corollary 1.2) |
| nadir VF 0.847, edge-on VF 0.258 @ 550 km | `environment.sphere_view_factor` |
| McCalip 335.75 &rarr; 342.10 K (+6.35 K) @ &beta;=90 | `mccalip_exact_vf` (doi:10.5281/zenodo.20695720) |
| Starcloud net 633.08 (published) / 584.76 (spectral) W/m&sup2; | `reference_architectures` (white paper v1.03) |

In [ ]:
_results = []

def check(name, got, expected, atol):
    ok = abs(got - expected) <= atol
    _results.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {got:.4f}  (expected {expected} +/- {atol})")

# AI1 operating points (equilibrium.equilibrium_temperature)
check("AI1 sustained T  [120 kW, 220 m^2, eps 0.91, Ts 220 K]",
      vapi.equilibrium_point(120_000, 220.0, 0.91, 220.0)["equilibrium_temperature_K"],
      337.1004, 1e-3)
check("AI1 peak T       [150 kW, 220 m^2, eps 0.91, Ts 220 K]",
      vapi.equilibrium_point(150_000, 220.0, 0.91, 220.0)["equilibrium_temperature_K"],
      353.1623, 1e-3)

# Megawatt area law (radiation.required_area)
check("1 MW area @ 293 K, eps 0.91, zero sink [emitting m^2]",
      vapi.required_area_point(1_000_000, 293.0, 0.91, 0.0)["required_emitting_area_m2"],
      2630.0, 0.5)

# Exact Earth view factor (environment.sphere_view_factor)
_vf = vapi.earth_view_factor_curve(550.0)
check("nadir view factor  @ 550 km", _vf["nadir_view_factor"], 0.847, 1e-3)
check("edge-on view factor @ 550 km", _vf["edge_on_view_factor"], 0.258, 3e-3)

# McCalip edge-on correction (mccalip_exact_vf)
_mc90 = vapi.mccalip_view_factor_comparison()["rows"][-1]
check("McCalip replicated eqtemp  @ beta 90", _mc90["eqtemp_mccalip_K"], 335.75, 0.05)
check("McCalip exact-VF   eqtemp  @ beta 90", _mc90["eqtemp_exact_K"], 342.10, 0.10)
check("McCalip edge-on correction delta_K", _mc90["delta_K"], 6.35, 0.05)

# Starcloud reference balances (reference_architectures)
_rt = vapi.reference_case_table()
_pub = next(r for r in _rt["rows"] if "published, as-written" in r["name"])
_spec = next(r for r in _rt["rows"] if "spectral separation" in r["name"])
check("Starcloud published net [W/m^2]", _pub["net_rejection_W_m2"], 633.08, 0.01)
check("Starcloud spectral  net [W/m^2]", _spec["net_rejection_W_m2"], 584.76, 0.01)

assert all(_results), "a baseline reproduction check FAILED"
print("\nAll baseline reproductions passed.")

## 4. Interactive radiator boundary controls & plots

The sliders below are **design variables** of the reduced-order boundary model.
Each plot is driven by a `visual_api` call (named in the code) that wraps the
engine; nothing is recomputed by hand.

> The two-sided **emitting area** = 2 &times; planform area is valid only when both
> faces see equal per-face sinks. The lumped effective sink `T_sink` is the
> papers' `T_s^eff = F^(1/4) * T_s`.

### 4.1 Required area vs radiator temperature
Higher rejection temperature &rarr; smaller radiator (the Lemma 1 area law
`A = Q / (eps*sigma*(T^4 - T_sink^4))`, via `radiation.required_area`).

In [ ]:
def plot_area_vs_temperature(Q_kW=120.0, emissivity=0.91, T_sink_K=220.0):
    res = vapi.area_temperature_curve(
        Q_kW * 1e3, emissivity=emissivity, T_sink_K=T_sink_K,
        t_min=T_sink_K + 5.0, t_max=650.0, num=300)
    fig = go.Figure()
    fig.add_scatter(x=res["x"], y=res["y"], mode="lines", name="emitting area")
    fig.add_scatter(x=res["x"], y=res["y_planform"], mode="lines",
                    name="planform area (= emitting / 2)")
    fig.update_layout(
        title=res["title"],
        xaxis_title=f"{res['x_label']} ({res['x_unit']})",
        yaxis_title=f"{res['y_label']} ({res['y_unit']})",
        template="plotly_white", height=420)
    return fig

interact(
    plot_area_vs_temperature,
    Q_kW=FloatSlider(value=120, min=10, max=1000, step=10, description="Q (kW)"),
    emissivity=FloatSlider(value=0.91, min=0.50, max=1.00, step=0.01, description="emissivity"),
    T_sink_K=FloatSlider(value=220, min=0, max=280, step=5, description="T_sink (K)"),
);

### 4.2 Net rejection vs radiator temperature
Net rejected flux per emitting face, `q = eps*sigma*(T^4 - T_sink^4)`
(via `radiation.net_flux`).

In [ ]:
def plot_net_rejection(emissivity=0.91, T_sink_K=220.0):
    res = vapi.net_rejection_curve(
        emissivity=emissivity, T_sink_K=T_sink_K,
        t_min=T_sink_K + 5.0, t_max=650.0, num=300)
    fig = go.Figure()
    fig.add_scatter(x=res["x"], y=res["y"], mode="lines", name="net flux")
    fig.update_layout(
        title=res["title"],
        xaxis_title=f"{res['x_label']} ({res['x_unit']})",
        yaxis_title=f"{res['y_label']} ({res['y_unit']})",
        template="plotly_white", height=420)
    return fig

interact(
    plot_net_rejection,
    emissivity=FloatSlider(value=0.91, min=0.50, max=1.00, step=0.01, description="emissivity"),
    T_sink_K=FloatSlider(value=220, min=0, max=280, step=5, description="T_sink (K)"),
);

### 4.3 Exact Earth view factor vs radiator tilt
The exact tilted-plate-to-sphere view factor (`environment.sphere_view_factor`),
no cosine approximation. Tilt 0&deg; = nadir-facing (max coupling), 90&deg; =
edge-on, 180&deg; = space-facing.

> **Limitation:** differential-element idealization &mdash; neglects finite-panel
> self-view and across-panel gradients (adequate for screening, not detailed panel
> design).

In [ ]:
def plot_view_factor(altitude_km=550.0):
    res = vapi.earth_view_factor_curve(altitude_km)
    fig = go.Figure()
    fig.add_scatter(x=res["x"], y=res["y"], mode="lines", name="exact view factor")
    fig.add_scatter(x=[0], y=[res["nadir_view_factor"]], mode="markers",
                    name=f"nadir = {res['nadir_view_factor']:.3f}", marker=dict(size=10))
    fig.add_scatter(x=[90], y=[res["edge_on_view_factor"]], mode="markers",
                    name=f"edge-on = {res['edge_on_view_factor']:.3f}", marker=dict(size=10))
    fig.update_layout(
        title=res["title"],
        xaxis_title=f"{res['x_label']} ({res['x_unit']})",
        yaxis_title=f"{res['y_label']} ({res['y_unit']})",
        template="plotly_white", height=420)
    return fig

interact(
    plot_view_factor,
    altitude_km=FloatSlider(value=550, min=300, max=2000, step=50, description="alt (km)"),
);

### 4.4 Effective sink vs beta angle
Radiatively-weighted orbit-mean effective sink `(<T_s_eff^4>)^(1/4)`, grid-free
(`sink.analytic_orbit_averaged_sink`).

> **Sun-shielded contract:** this effective-sink model **omits direct solar flux**
> on the radiator face and is valid only when that face is sun-shielded (anti-solar
> attitude or external shade). The engine requires `assume_sun_shielded=True`.
>
> **&beta; = 90&deg; is model-limited, not physical:** the subpoint albedo factor
> `cos(beta)/pi` nulls at the dawn-dusk endpoint. That is a limitation of the
> subpoint-albedo model (true disk-integrated albedo is nonzero there), flagged
> with a red marker &mdash; **not** a claim that the radiator sees no reflected sun.

In [ ]:
def plot_effective_sink_vs_beta(altitude_km=550.0, tilt_deg=0.0,
                                emissivity=0.91, solar_absorptivity=0.20):
    res = vapi.effective_sink_sweep(
        altitude_km, betas_deg=tuple(range(0, 91, 5)), tilt_deg=tilt_deg,
        emissivity=emissivity, solar_absorptivity=solar_absorptivity)
    betas = [r["beta_deg"] for r in res["rows"]]
    sinks = [r["orbit_averaged_sink_K"] for r in res["rows"]]
    limited = [r["albedo_model_limited"] for r in res["rows"]]
    fig = go.Figure()
    fig.add_scatter(x=betas, y=sinks, mode="lines+markers", name="orbit-mean effective sink")
    lb = [b for b, l in zip(betas, limited, strict=True) if l]
    ls = [s for s, l in zip(sinks, limited, strict=True) if l]
    if lb:
        fig.add_scatter(x=lb, y=ls, mode="markers", name="albedo model-limited (beta=90)",
                        marker=dict(color="red", size=12, symbol="x"))
    fig.update_layout(
        title=res["title"],
        xaxis_title="Orbit beta angle (deg)",
        yaxis_title="Radiatively-weighted orbit-mean effective sink (K)",
        template="plotly_white", height=420)
    return fig

interact(
    plot_effective_sink_vs_beta,
    altitude_km=FloatSlider(value=550, min=300, max=2000, step=50, description="alt (km)"),
    tilt_deg=FloatSlider(value=0, min=0, max=180, step=10, description="tilt (deg)"),
    emissivity=FloatSlider(value=0.91, min=0.50, max=1.00, step=0.01, description="emissivity"),
    solar_absorptivity=FloatSlider(value=0.20, min=0.0, max=1.0, step=0.05,
                                   description="alpha_s (assumed)"),
);

### 4.5 McCalip heuristic vs exact view factor (edge-on correction)
McCalip's own heat balance held fixed; only the per-face Earth view factor is
swapped (his cos-tilt heuristic + 5% edge-on floor &rarr; the exact
tilted-plate-to-sphere integral). Left: per-face view factor; right: the resulting
equilibrium-temperature correction (`mccalip_exact_vf.correction_table_vs_beta`).

> This is a **replication + geometry correction** of an external model, **not** a
> validation of that model against reality. Replication keeps McCalip's truncated
> `sigma = 5.67e-8`, `T_space = 3 K`, and 72-point orbit average.

In [ ]:
def plot_mccalip(altitude_km=550.0):
    res = vapi.mccalip_view_factor_comparison(altitude_km=altitude_km)
    rows = res["rows"]
    betas = [r["beta_deg"] for r in rows]

    fig_vf = go.Figure()
    fig_vf.add_scatter(x=betas, y=[r["vf_side_a_mccalip_heuristic"] for r in rows],
                       mode="lines+markers", name="McCalip heuristic (side A)")
    fig_vf.add_scatter(x=betas, y=[r["vf_side_a_exact"] for r in rows],
                       mode="lines+markers", name="exact per-face (side A)")
    fig_vf.update_layout(
        title=f"Per-face Earth view factor: heuristic vs exact ({res['altitude_km']:.0f} km)",
        xaxis_title="Orbit beta angle (deg)", yaxis_title="View factor (-)",
        template="plotly_white", height=380)

    fig_t = go.Figure()
    fig_t.add_scatter(x=betas, y=[r["eqtemp_mccalip_K"] for r in rows],
                      mode="lines+markers", name="McCalip replicated (K)")
    fig_t.add_scatter(x=betas, y=[r["eqtemp_exact_K"] for r in rows],
                      mode="lines+markers", name="exact-VF (K)")
    fig_t.add_scatter(x=betas, y=[r["delta_K"] for r in rows],
                      mode="lines+markers", name="delta (exact - McCalip, K)", yaxis="y2")
    fig_t.update_layout(
        title=f"Equilibrium-temperature correction vs beta ({res['altitude_km']:.0f} km)",
        xaxis_title="Orbit beta angle (deg)",
        yaxis=dict(title="Equilibrium temperature (K)"),
        yaxis2=dict(title="delta (K)", overlaying="y", side="right"),
        template="plotly_white", height=380)

    display(fig_vf)
    display(fig_t)

interact(
    plot_mccalip,
    altitude_km=FloatSlider(value=550, min=300, max=2000, step=50, description="alt (km)"),
);

### 4.6 Orbital transient waveform
One-node transient `C dT/dt = q_load - eps*sigma*(T^4 - T_s_eff(t)^4)` marched to a
periodic steady state (`transient.simulate`). The panel lags and ripples as the
sink swings; the dashed line is the steady solution at the radiatively-weighted
average sink. The steady, averaged-load assumption under-predicts the **peak** (a
positive `peak_excess`), while the arithmetic mean sits at or below steady (Jensen).

> Same **sun-shielded contract** as the sink model. Areal heat capacity comes from
> a named representative material build (`transient.REPRESENTATIVE_BUILDS`), so it
> is *derived* from a documented stack rather than assumed.

In [ ]:
_builds = list(_transient.REPRESENTATIVE_BUILDS)

def show_transient(beta_deg=0.0, q_load_W_m2=545.0,
                   build="integrated_compute_radiator", steps_per_orbit=720):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = vapi.transient_orbit_case(
            550.0, beta_deg, q_load_W_m2, build_name=build,
            n_orbits=40, steps_per_orbit=steps_per_orbit)
    s = res["summary"]
    fig = go.Figure()
    fig.add_scatter(x=res["t_min"], y=res["T_panel_K"], mode="lines", name="panel T")
    fig.add_scatter(x=res["t_min"], y=res["T_sink_K"], mode="lines",
                    name="effective sink T", line=dict(dash="dot"))
    fig.add_hline(y=s["steady_avg_sink_K"], line_dash="dash",
                  annotation_text=f"steady @ avg sink = {s['steady_avg_sink_K']:.1f} K")
    fig.update_layout(
        title=res["title"],
        xaxis_title=f"{res['x_label']} ({res['x_unit']})",
        yaxis_title=f"{res['y_label']} ({res['y_unit']})",
        template="plotly_white", height=440)
    display(fig)
    print(f"C = {res['areal_heat_capacity_J_m2K']:.0f} J/m^2/K   "
          f"tau/period = {s['tau_over_period']:.2f}   converged = {s['converged']}")
    print(f"peak {s['transient_peak_K']:.2f} K | mean {s['transient_mean_K']:.2f} K | "
          f"steady {s['steady_avg_sink_K']:.2f} K | peak excess {s['peak_excess_over_steady_K']:.2f} K "
          f"| swing {s['swing_K']:.2f} K")
    if res["meta"]["warnings"]:
        for w in res["meta"]["warnings"]:
            print("WARNING:", w)

interact(
    show_transient,
    beta_deg=FloatSlider(value=0, min=0, max=90, step=5, description="beta (deg)"),
    q_load_W_m2=FloatSlider(value=545, min=100, max=1500, step=25, description="q (W/m^2)"),
    build=Dropdown(options=_builds, value="integrated_compute_radiator", description="build"),
    steps_per_orbit=IntSlider(value=720, min=360, max=2000, step=180, description="steps/orbit"),
);

## 5. Reference cases (labelled by provenance / convention)

Only package-supported cases are computed here; each row is labelled and its value
provenance recorded. **This table does not rank architectures**
(`ranking_performed = False`): as-published cases are never a ranking basis, and
AI1's unpublished solar absorptivity leaves its harmonized albedo unresolved (left
`None`, never invented).

**Label legend** &mdash;
<span style="background:#dbeafe;padding:1px 5px;border-radius:3px">published</span>
<span style="background:#dcfce7;padding:1px 5px;border-radius:3px">harmonized</span>
<span style="background:#fef9c3;padding:1px 5px;border-radius:3px">sensitivity</span>
<span style="background:#fee2e2;padding:1px 5px;border-radius:3px">unsupported</span>
<span style="background:#ede9fe;padding:1px 5px;border-radius:3px">future</span>

In [ ]:
_LABEL_COLORS = {
    "published": "#dbeafe", "harmonized": "#dcfce7", "sensitivity": "#fef9c3",
    "unsupported": "#fee2e2", "future": "#ede9fe",
}

def render_reference_table(load_kW=120.0, harmonized_beta_deg=45.0):
    rt = vapi.reference_case_table(load_W=load_kW * 1e3, harmonized_beta_deg=harmonized_beta_deg)

    def fmt(v):
        if v is None:
            return "&mdash;"
        if isinstance(v, bool):
            return "yes" if v else "no"
        if isinstance(v, float):
            return f"{v:.2f}"
        return str(v)

    cols = ["name", "label", "rank_eligible", "radiator_temperature_K",
            "net_rejection_W_m2", "notes"]
    headers = ["Case", "Label", "Rank-eligible", "T_rad (K)", "net (W/m^2)", "Notes"]
    html = ["<table style='border-collapse:collapse;font-size:12px'>"]
    html.append("<tr>" + "".join(
        f"<th style='border:1px solid #ccc;padding:4px;text-align:left'>{h}</th>"
        for h in headers) + "</tr>")
    for r in rt["rows"]:
        color = _LABEL_COLORS.get(r["label"], "#fff")
        cells_html = []
        for c in cols:
            style = "border:1px solid #ccc;padding:4px;vertical-align:top"
            if c == "label":
                style += f";background:{color};font-weight:bold"
            if c == "notes":
                style += ";max-width:360px"
            cells_html.append(f"<td style='{style}'>{fmt(r.get(c))}</td>")
        html.append("<tr>" + "".join(cells_html) + "</tr>")
    html.append("</table>")
    display(HTML("".join(html)))
    print(f"ranking_performed = {rt['ranking_performed']}")
    print("source functions:", ", ".join(rt["meta"]["source_functions"]))

interact(
    render_reference_table,
    load_kW=FloatSlider(value=120, min=50, max=1000, step=10, description="load (kW)"),
    harmonized_beta_deg=FloatSlider(value=45, min=0, max=90, step=15, description="harm. beta"),
);

### 5.1 Biswas / Suncatcher &mdash; recorded future reference (not integrated)

The Biswas/Suncatcher model appears in the table as a **`future`** row only. It is
**recorded, not reproduced** here:

* pinned to **release `v1.2`** (author short commit `23053beeff53`; the full
  40-char SHA is to be resolved and recorded);
* a **two-phase heat-pipe** architecture &mdash; it cannot be a ranked Phase B
  Stage-1 case (Stage 1 is single-phase) and becomes a **Stage-2 benchmark** later;
* **not ranked** against the other cases;
* author cross-check is **source-author review, not** independent external
  validation; public documentation requires project-director approval.

Building the full Biswas reproduction is **out of scope** for B0.5.

## 6. Limitations and next steps

**Explicit limitations of what is shown above:**

* **Direct solar is omitted.** The effective-sink and transient outputs use the
  cold-side model, valid only under the **sun-shielded contract**
  (`assume_sun_shielded=True`). A sun-facing/tilted face that sees direct sun
  violates it; a sourced/parametric direct-solar extension is required otherwise.
* **`sink` requires `assume_sun_shielded=True`.** The engine hard-rejects any other
  value; these visualizations therefore always carry the shielding assumption.
* **&beta; = 90&deg; subpoint-albedo null is a model limitation, not a physical
  conclusion.** The true disk-integrated albedo is nonzero at a terminator orbit;
  `sink.disk_integrated_albedo_factor` is intentionally not implemented.
* **Phase B chip-to-radiator transport is not included.** No pumped loop, no
  coupled solver, no two-phase transport. That waits for Phase B (B4/B5).
* Cross-model comparison (McCalip) is **replication + correction**, not validation;
  as-published cases are **not** a ranking.

**Next steps (future work, not this task):**

* A direct-solar extension (sourced `alpha_s` + incidence geometry) to relax the
  shielded contract.
* A disk-integrated albedo model to remove the &beta;=90 null.
* The pinned Biswas/Suncatcher `v1.2` reproduction as a Stage-2 (two-phase)
  benchmark, on its own track.
* Phase B coupled chip-to-radiator transport visualization (B4/B5).